**Author:** Ashutosh Jayant  
**Project:** India State Competitiveness Index (ISCI)

# 05 - Validation and sensitivity

Tests how stable the ranking is: different unemployment definitions, the coverage rule,
and rank correlation between the variants.

## Setup
Load the master table and the scoring code from src/scoring.py, so the tests use the same method as Notebook 04.

In [1]:
import sys
from pathlib import Path
import pandas as pd

root = Path.cwd()
if not (root / "src").exists():
    root = root.parent
sys.path.insert(0, str(root))

from src import scoring

master = pd.read_csv(root / "data" / "processed" / "master_features.csv", index_col=0)
print(master.shape)
print(scoring.INDICATORS)

(36, 14)
['ger', 'life_expectancy', 'unemployment_rate', 'cd_ratio', 'per_capita_power', 'td_losses', 'road_density', 'factory_density', 'msme_density', 'manufacturing_share', 'investment_rate']


## Three unemployment versions
Build the ranking three times, using rural, urban and the average, and put the ranks side by side.

In [2]:
def rank_with(unemp):
    df = master.copy()
    df["unemployment_rate"] = unemp
    scores = scoring.indicator_scores(df[scoring.INDICATORS])
    return scoring.build_index(scores)

variants = {
    "rural":   master["unemp_rural"],
    "urban":   master["unemp_urban"],
    "average": master[["unemp_rural", "unemp_urban"]].mean(axis=1),
}

rankings = {name: rank_with(u) for name, u in variants.items()}

compare = pd.DataFrame({f"{name}_rank": r["Rank"] for name, r in rankings.items()})
compare = compare.sort_values("average_rank")
print(compare.to_string())

                                      rural_rank  urban_rank  average_rank
entity                                                                    
Goa                                            3           1             1
Tamil Nadu                                     1           2             2
Gujarat                                        2           3             3
Puducherry                                     4           4             4
Telangana                                      5           5             5
Andhra Pradesh                                 7           7             6
Punjab                                         9           6             7
Maharashtra                                    6           8             8
Himachal Pradesh                               8           9             9
Haryana                                       10          10            10
Uttarakhand                                   11          12            11
Karnataka                

## Rank correlation
Measure how close the three rankings are with Spearman correlation. Near 1 means almost the same order.

In [3]:
corr = compare.dropna().corr(method="spearman")
print(corr.round(4).to_string())

              rural_rank  urban_rank  average_rank
rural_rank        1.0000      0.9913        0.9960
urban_rank        0.9913      1.0000        0.9967
average_rank      0.9960      0.9967        1.0000


## Top and bottom stability
Check whether the same states stay in the top 10 and bottom 10 across the three versions.

In [4]:
def members(col, lo, hi):
    r = compare[col]
    return set(r[(r >= lo) & (r <= hi)].index)

for label, lo, hi in [("Top 10", 1, 10), ("Bottom 10", 24, 33)]:
    sets = {name: members(f"{name}_rank", lo, hi) for name in variants}
    common = set.intersection(*sets.values())
    moving = set.union(*sets.values()) - common
    print(f"{label}: {len(common)} of 10 states appear in all three versions")
    if moving:
        print("  moves in or out at the boundary:", sorted(moving))
    print()

Top 10: 10 of 10 states appear in all three versions

Bottom 10: 9 of 10 states appear in all three versions
  moves in or out at the boundary: ['Chhattisgarh', 'Madhya Pradesh']



## Coverage rule check
Rebuild the ranking with the rule set to 7, 8 and 9, and see which states enter or leave.

In [5]:
df = master.copy()
df["unemployment_rate"] = master[["unemp_rural", "unemp_urban"]].mean(axis=1)
scores = scoring.indicator_scores(df[scoring.INDICATORS])

base = scoring.build_index(scores, min_indicators=8)
base_ranked = set(base.index[base["Rank"].notna()])

for k in (7, 8, 9):
    idx = scoring.build_index(scores, min_indicators=k)
    ranked = set(idx.index[idx["Rank"].notna()])
    diff = sorted(ranked ^ base_ranked)
    tag = "baseline" if k == 8 else f"vs 8/11: {diff}"
    print(f"min {k}/11 -> {len(ranked)} states ranked   {tag}")

min 7/11 -> 34 states ranked   vs 8/11: ['Ladakh']
min 8/11 -> 33 states ranked   baseline
min 9/11 -> 32 states ranked   vs 8/11: ['Andaman & Nicobar Islands']


## Min-Max vs Z-score
Score the states again with Z-score and compare with the Min-Max ranking.

In [6]:
def zscore(col, negative=False):
    z = (col - col.mean()) / col.std()
    return -z if negative else z

df = master.copy()
df["unemployment_rate"] = master[["unemp_rural", "unemp_urban"]].mean(axis=1)

z = pd.DataFrame({c: zscore(df[c], c in scoring.NEGATIVE) for c in scoring.INDICATORS})
z_available = z.notna().sum(axis=1)
z_overall = z.mean(axis=1)
z_overall[z_available < 8] = float("nan")
z_rank = z_overall.rank(ascending=False, method="min").astype("Int64")

mm = scoring.build_index(scoring.indicator_scores(df[scoring.INDICATORS]), 8)

comp = pd.DataFrame({"minmax_rank": mm["Rank"], "zscore_rank": z_rank}).dropna()
print("Spearman (Min-Max vs Z-score):", round(comp.corr(method="spearman").iloc[0, 1], 4))
print()
print(comp.sort_values("minmax_rank").to_string())

Spearman (Min-Max vs Z-score): 0.9886

                           minmax_rank  zscore_rank
entity                                             
Goa                                  1            1
Tamil Nadu                           2            2
Gujarat                              3            3
Puducherry                           4            4
Telangana                            5            5
Andhra Pradesh                       6            6
Punjab                               7            8
Maharashtra                          8            7
Himachal Pradesh                     9            9
Haryana                             10           10
Uttarakhand                         11           13
Karnataka                           12           12
Delhi                               13           11
Odisha                              14           14
Kerala                              15           15
Sikkim                              16           16
West Bengal              

## Robustness summary

Three checks were carried out, and all gave the same picture.

Unemployment: using rural, urban or the simple average produced almost the same ranking.
The Spearman correlation between the three versions was 0.99 or higher. The top 10 states
stayed the same in all three. Only one state changed near the bottom, where Chhattisgarh
and Madhya Pradesh swapped places around rank 23.

Coverage rule: changing the rule from 8 of 11 had very little effect. At 7 of 11, Ladakh
entered the ranking. At 9 of 11, Andaman & Nicobar Islands dropped out. No other state
changed.

Normalization: Min-Max and Z-score produced almost identical rankings, with a Spearman
correlation of 0.99. The highest-ranked and lowest-ranked states stayed the same, and
only a few middle-ranked states moved by a small number of positions.

The ranking stayed stable under all three checks. This suggests the Version 1.0 results
are not driven by a particular choice of unemployment measure, coverage rule or
normalization method.